In [0]:
SOURCE = "etlpipelines.default.silver_global_superstore"
SCHEMA = "etlpipelines.goldetl"

spark.sql("CREATE SCHEMA IF NOT EXISTS etlpipelines.goldetl")

queries = {
    "gold_geographic_performance": """
SELECT country, state, city, region, market,
 SUM(sales) total_sales, SUM(profit) total_profit, SUM(quantity) total_quantity
FROM {src} GROUP BY country, state, city, region, market""",

"gold_product_performance": """
SELECT product_id, product_name, category, sub_category,
 COUNT(DISTINCT order_id) total_orders, SUM(quantity) total_quantity,
 SUM(sales) total_sales, SUM(profit) total_profit, AVG(discount) avg_discount,
 ROUND(SUM(profit)/NULLIF(SUM(sales),0)*100,2) profit_margin_pct
FROM {src} GROUP BY product_id, product_name, category, sub_category""",

"gold_category_subcategory": """
SELECT category, sub_category, COUNT(DISTINCT product_id) total_products,
 COUNT(DISTINCT order_id) total_orders, SUM(quantity) total_quantity,
 SUM(sales) total_sales, SUM(profit) total_profit, AVG(discount) avg_discount
FROM {src} GROUP BY category, sub_category""",

"gold_customer_performance": """
SELECT customer_id, customer_name, segment, COUNT(DISTINCT order_id) total_orders,
 MIN(order_date) first_order_date, MAX(order_date) last_order_date,
 SUM(sales) lifetime_sales, SUM(profit) lifetime_profit, SUM(quantity) total_quantity,
 ROUND(SUM(sales)/NULLIF(COUNT(DISTINCT order_id),0),2) avg_order_value
FROM {src} GROUP BY customer_id, customer_name, segment""",

"gold_segment_performance": """
SELECT segment, COUNT(DISTINCT customer_id) total_customers,
 COUNT(DISTINCT order_id) total_orders, SUM(sales) total_sales,
 SUM(profit) total_profit, SUM(quantity) total_quantity,
 ROUND(AVG(discount),3) avg_discount,
 ROUND(SUM(profit)/NULLIF(SUM(sales),0)*100,2) profit_margin_pct
FROM {src} GROUP BY segment""",

"gold_shipping_performance": """
SELECT ship_mode, COUNT(DISTINCT order_id) total_orders, SUM(sales) total_sales,
 SUM(profit) total_profit, SUM(shipping_cost) total_shipping_cost,
 AVG(shipping_cost) avg_shipping_cost,
 AVG(DATEDIFF(ship_date,order_date)) avg_shipping_days
FROM {src} GROUP BY ship_mode""",

"gold_order_priority": """
SELECT order_priority, COUNT(DISTINCT order_id) total_orders, SUM(sales) total_sales,
 SUM(profit) total_profit, SUM(quantity) total_quantity,
 AVG(shipping_cost) avg_shipping_cost
FROM {src} GROUP BY order_priority""",

"gold_discount_analysis": """
SELECT CASE WHEN discount IS NULL THEN 'Unknown'
 WHEN discount=0 THEN '0 - No Discount'
 WHEN discount<=0.10 THEN '1 - Up to 10%'
 WHEN discount<=0.20 THEN '2 - 10% to 20%'
 WHEN discount<=0.30 THEN '3 - 20% to 30%'
 ELSE '4 - Above 30%' END discount_band,
 COUNT(*) total_order_lines, COUNT(DISTINCT order_id) total_orders,
 SUM(quantity) total_quantity, SUM(sales) total_sales, SUM(profit) total_profit,
 AVG(discount) avg_discount
FROM {src}
GROUP BY CASE WHEN discount IS NULL THEN 'Unknown'
 WHEN discount=0 THEN '0 - No Discount'
 WHEN discount<=0.10 THEN '1 - Up to 10%'
 WHEN discount<=0.20 THEN '2 - 10% to 20%'
 WHEN discount<=0.30 THEN '3 - 20% to 30%'
 ELSE '4 - Above 30%' END""",

"gold_profitability_analysis": """
SELECT region, market, category, sub_category, SUM(sales) total_sales,
 SUM(profit) total_profit, SUM(quantity) total_quantity,
 ROUND(SUM(profit)/NULLIF(SUM(sales),0)*100,2) profit_margin_pct,
 CASE WHEN SUM(profit)<0 THEN 'Loss' WHEN SUM(profit)=0 THEN 'Break-even'
 ELSE 'Profitable' END profitability_status
FROM {src} GROUP BY region, market, category, sub_category""",

"gold_top_products": """
WITH p AS (
 SELECT product_id, product_name, category, sub_category, SUM(sales) total_sales,
 SUM(profit) total_profit, SUM(quantity) total_quantity
 FROM {src} GROUP BY product_id, product_name, category, sub_category
), r AS (
 SELECT *, DENSE_RANK() OVER(PARTITION BY category ORDER BY total_sales DESC) sales_rank
 FROM p
)
SELECT * FROM r WHERE sales_rank<=10""",

"gold_customer_retention": """
WITH c AS (
 SELECT customer_id, MAX(customer_name) customer_name, MAX(segment) segment,
 COUNT(DISTINCT order_id) total_orders, MIN(order_date) first_order_date,
 MAX(order_date) last_order_date, SUM(sales) total_sales, SUM(profit) total_profit
 FROM {src} GROUP BY customer_id
)
SELECT *, DATEDIFF(last_order_date,first_order_date) customer_lifetime_days,
 CASE WHEN total_orders=1 THEN 'One-time Customer' ELSE 'Repeat Customer' END customer_type
FROM c""",

"gold_order_details": """
SELECT row_id, order_id, order_date, ship_date, ship_mode, customer_id, customer_name,
 segment, country, city, state, region, market, product_id, product_name, category,
 sub_category, sales, quantity, discount, profit, shipping_cost, order_priority
FROM {src}"""
}

for table, query in queries.items():
    spark.sql(f"CREATE OR REPLACE TABLE {SCHEMA}.{table} USING DELTA AS " +
              query.format(src=SOURCE))
    print(f"Created {SCHEMA}.{table}")

print("\nGold table row counts:")
for table in queries:
    full_name = f"{SCHEMA}.{table}"
    print(f"{full_name}: {spark.table(full_name).count()} rows")

print(f"\nSUCCESS: Created and checked {len(queries)} Gold tables.")



Created etlpipelines.goldetl.gold_geographic_performance
Created etlpipelines.goldetl.gold_product_performance
Created etlpipelines.goldetl.gold_category_subcategory
Created etlpipelines.goldetl.gold_customer_performance
Created etlpipelines.goldetl.gold_segment_performance
Created etlpipelines.goldetl.gold_shipping_performance
Created etlpipelines.goldetl.gold_order_priority
Created etlpipelines.goldetl.gold_discount_analysis
Created etlpipelines.goldetl.gold_profitability_analysis
Created etlpipelines.goldetl.gold_top_products
Created etlpipelines.goldetl.gold_customer_retention
Created etlpipelines.goldetl.gold_order_details

Gold table row counts:
etlpipelines.goldetl.gold_geographic_performance: 3819 rows
etlpipelines.goldetl.gold_product_performance: 10768 rows
etlpipelines.goldetl.gold_category_subcategory: 17 rows
etlpipelines.goldetl.gold_customer_performance: 1590 rows
etlpipelines.goldetl.gold_segment_performance: 3 rows
etlpipelines.goldetl.gold_shipping_performance: 4 rows